In [234]:
from sklearn.datasets import load_boston
import numpy as np
import matplotlib.pyplot as plt
import warnings
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures

## Avoid printing out warnings

with warnings.catch_warnings():
     warnings.filterwarnings("ignore")
     X, y = load_boston(return_X_y=True)

print(X.shape)
print(y.shape)
# print(X[:5])

(506, 13)
(506,)


In [235]:
# Split data into train and test data, train = 80% and test = 20%
def split(X, y):
    np.random.seed(42)
    indices = np.random.permutation(X.shape[0])
    test_size = int(X.shape[0]*.20)
    test_indices, train_indices = indices[:test_size], indices[test_size:]
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]
    
X_train, X_test, y_train, y_test = split(X,y)
print(len(X_train))
print(len(X_test))

405
101


# Linear Regression (Closed Form)

In [247]:
#using the normal equation theta = (X**T*X)**(-1)*X**T*y which was derived from the derivative of our loss function
def add_bias(X):
    return np.c_[np.ones(X.shape[0]),X]
def normal_equation(X,y):
    theta = np.linalg.inv(X.T @ X) @ X.T @ y
    return theta
    
def linear_regression_model(X_train,y_train, X_test, y_test):
    X_train_bias = add_bias(X_train)
    X_test_bias = add_bias(X_test)

    theta = normal_equation(X_train_bias, y_train)

    #calcualting prediction
    y_train_pred = X_train_bias @ theta
    y_test_pred = X_test_bias @ theta
    # print(y_train_pred)

    #mse calculation
    mse_train = np.mean((y_train - y_train_pred)**2)
    mse_test = np.mean((y_test - y_test_pred)**2)

    return mse_train, mse_test

mse_train, mse_test = linear_regression_model(X_train,y_train, X_test, y_test)

print("80+20 split of train and test data")
print(f'mse_train: {mse_train}')
print(f'mse_test: {mse_test}')


80+20 split of train and test data
mse_train: 21.610941753516673
mse_test: 24.396833763216957


In [248]:
# KFold cross-validation
def k_fold_split(X,y,k):
    np.random.seed(42)
    indices = np.random.permutation(X.shape[0])
    fold_size = X.shape[0]//k  #data per fold
    folds = []
    for i in range(k):
        start = i*fold_size
        end = (i+1)*fold_size
        folds.append(indices[start:end])
        
    return folds
    
def k_fold_cross_validation(X,y, k=10):
    folds = k_fold_split(X,y,k)
    train_erros, test_errors = [], []

    for i in range(k):
        test_indices = folds[i]
        train_indices = np.concatenate([folds[j] for j in range(k) if j != i])

        X_train, X_test = X[train_indices], X[test_indices]
        y_train, y_test = y[train_indices], y[test_indices]
        
        mse_train, mse_test = linear_regression_model(X_train,y_train, X_test, y_test)
        train_errors.append(mse_train)
        test_errors.append(mse_test)

    mse_train_final = np.mean(train_errors)
    mse_test_final = np.mean(test_errors)

    return mse_train_final, mse_test_final
    

mse_train, mse_test = k_fold_cross_validation(X,y)
print("Using kFold cross validation where k=10")
print(f'mse_train: {mse_train}')
print(f'mse_test: {mse_test}')

    

Using kFold cross validation where k=10
mse_train: 21.07464382923407
mse_test: 23.707896390630758


# Ridge Regression

In [249]:
def ridge_regression_eq(X,y,lambda_curr):
    I = np.eye(X.shape[1]) #identity matrix
    theta_ridge = np.linalg.inv(X.T @ X + lambda_curr*I) @ X.T @ y
    return theta_ridge

def k_fold_ridge_regression(X,y, lambdas, k=10):
    selected_lambda = None
    selected_mse_test = np.inf
    mse_train = 0
    
    for lambda_curr in lambdas:
        folds = k_fold_split(X,y,k)
        train_erros, test_errors = [], []

        for i in range(k):
            test_indices = folds[i]
            train_indices = np.concatenate([folds[j] for j in range(k) if j != i])
           
            #getting X and ys
            X_train, X_test = X[train_indices], X[test_indices]
            y_train, y_test = y[train_indices], y[test_indices] 
    
            #adding bias
            X_train_bias = add_bias(X_train)
            X_test_bias = add_bias(X_test)
        
            theta = ridge_regression_eq(X_train_bias, y_train, lambda_curr)
        
            #calcualting prediction
            y_train_pred = X_train_bias @ theta
            y_test_pred = X_test_bias @ theta
        
            #mse calculation
            mse_train = np.mean((y_train - y_train_pred)**2)
            mse_test = np.mean((y_test - y_test_pred)**2)
            train_errors.append(mse_train)
            test_errors.append(mse_test)

        #average mse per lambda
        mse_test_avg = np.mean(test_errors)
        mse_train_avg = np.mean(train_errors)
        

        #comparing to get the selected lambda
        if mse_test_avg < selected_mse_test:
            selected_lambda =lambda_curr
            selected_mse_test = mse_test_avg
            mse_train = mse_train_avg

        return selected_lambda, selected_mse_test, mse_train
    

lambda_values = np.logspace(1,7, num=13)
# print(lamda_values)
best_lambda, lowest_mse_test, mse_train = k_fold_ridge_regression(X,y,lambda_values)
print(f'The best lambda is {best_lambda} and the mse it gives is {lowest_mse_test}')


The best lambda is 10.0 and the mse it gives is 25.582723064611297


In [250]:
#Using the best lambda 10 to get the mse_test and mse_train
#Will do that directly from previous codee
print(f'The lambda here is {best_lambda}')
print(f'mse for test set is {lowest_mse_test}')
print(f'mse for train set is {mse_train}')

The lambda here is 10.0
mse for test set is 25.582723064611297
mse for train set is 21.14936798962705


# Polynomial Transformation

In [251]:
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# performing kFold ridge regression on polynomial transformed features
best_lambda, lowest_mse_test, mse_train = k_fold_ridge_regression(X_poly,y,lambda_values)
print(f'The best lambda here is {best_lambda}')
print(f'mse for test set is {lowest_mse_test}')
print(f'mse for train set is {mse_train}')


The best lambda here is 10.0
mse for test set is 13.953776429059392
mse for train set is 20.803123774474955


# Gradient Descent

In [252]:
import numpy as np

def cost(X, y, theta):
    m = len(y)
    predictions = X @ theta
    cost = (1/(2*m)) * np.sum((predictions - y)**2)
    return cost

def gradient_descent(X, y, theta, alpha, iterations):
    m = len(y)
    costs_observed = []

    for i in range(iterations):
        gradients = (1/m) * (X.T @ (X @ theta - y))
        theta = theta - alpha * gradients
        costs_observed.append(cost(X, y, theta))

        # Print iteration update
        if i % 5000 == 0: 
            print(f'Iteration {i}, Cost: {costs_observed[-1]}')

    return theta, costs_observed

X_train_bias = add_bias(X_train)
X_test_bias = add_bias(X_test)
# print(X_bias.shape[1])

# Initialize theta
theta_initial = np.zeros(X_train_bias.shape[1])
alpha = 0.000006  #fixed after multiple trials, learning rate
iterations = 60000  # fixed after multiple trials

theta_final, costs_observed = gradient_descent(X_train_bias, y_train, theta_initial, alpha, iterations)
print(f'Final Theta Values: {theta_final}')

Iteration 0, Cost: 232.07777581798152
Iteration 5000, Cost: 27.067670798704413
Iteration 10000, Cost: 24.377631689243536
Iteration 15000, Cost: 23.231436877372523
Iteration 20000, Cost: 22.50339542114574
Iteration 25000, Cost: 21.942934302348405
Iteration 30000, Cost: 21.469365163530817
Iteration 35000, Cost: 21.047905444055097
Iteration 40000, Cost: 20.66104800168087
Iteration 45000, Cost: 20.29930849561336
Iteration 50000, Cost: 19.95730451366859
Iteration 55000, Cost: 19.63184197730871
Final Theta Values: [ 1.29444468e-01 -1.03696184e-01  7.99309746e-02  4.00647518e-04
  1.07340746e-01  5.84174334e-02  1.46740488e+00  9.44501916e-02
 -6.79013126e-02  6.56033809e-02 -1.92531708e-03  3.91838707e-01
  2.67626323e-02 -8.21452266e-01]


In [254]:
#calcualting prediction
y_train_pred = X_train_bias @ theta_final
y_test_pred = X_test_bias @ theta_final

#mse calculation
mse_train = np.mean((y_train - y_train_pred)**2)
mse_test = np.mean((y_test - y_test_pred)**2)
print(f'mse_train: {mse_train}')
print(f'mse_test: {mse_test}')

mse_train: 38.641967588989296
mse_test: 38.569806862078515


# Lasso Regression

In [255]:
def compute_cost_lasso(X, y, theta, alpha):
    m = len(y)
    predictions = X @ theta
    mse = (1/(2*m)) * np.sum((predictions - y)**2)
    l1_penalty = alpha * np.sum(np.abs(theta))
    return mse + l1_penalty

def lasso_coordinate_descent(X, y, alpha, iterations):
    m, n = X.shape
    theta = np.zeros(n)

    for it in range(iterations):
        for j in range(n):
            X_j = X[:, j]
            residual = y - (X @ theta) + (theta[j] * X_j)
            rho = X_j @ residual
            
            if rho < -alpha:
                theta[j] = (rho + alpha) / (X_j @ X_j)
            elif rho > alpha:
                theta[j] = (rho - alpha) / (X_j @ X_j)
            else:
                theta[j] = 0

        if it % 1000 == 0:
            cost = compute_cost_lasso(X, y, theta, alpha)
            print(f"Iteration no {it}, Cost {cost}")

    return theta

X_train_bias = add_bias(X_train)
X_test_bias = add_bias(X_test)
# print(X_bias.shape[1])

alpha = 0.00001  # regularization strength, fixed after multiple trial
iterations = 10000  # Number of iterations

theta_lasso = lasso_coordinate_descent(X_train_bias, y_train, alpha, iterations)
print("\nFinal Theta Values:", theta_lasso)

Iteration no 0, Cost 28.812555904285645
Iteration no 1000, Cost 10.80609231419755
Iteration no 2000, Cost 10.806048296320496
Iteration no 3000, Cost 10.80604807365256
Iteration no 4000, Cost 10.80604806743748
Iteration no 5000, Cost 10.806048067139496
Iteration no 6000, Cost 10.806048067124653
Iteration no 7000, Cost 10.806048067123916
Iteration no 8000, Cost 10.80604806712388
Iteration no 9000, Cost 10.806048067123875

Final Theta Values: [ 3.01452021e+01 -1.13053400e-01  3.07557583e-02  3.83749780e-02
  2.78643281e+00 -1.70055211e+01  4.43604827e+00 -5.98877573e-03
 -1.44796027e+00  2.64769434e-01 -1.08061167e-02 -9.13264902e-01
  1.23437847e-02 -5.08514802e-01]


In [256]:
#calcualting prediction
y_train_pred = X_train_bias @ theta_lasso
y_test_pred = X_test_bias @ theta_lasso

#mse calculation
mse_train = np.mean((y_train - y_train_pred)**2)
mse_test = np.mean((y_test - y_test_pred)**2)
print(f'mse_train: {mse_train}')
print(f'mse_test: {mse_test}')

mse_train: 21.610941753517523
mse_test: 24.39683531339234


# Elastic Net

In [257]:
def compute_cost_elastic_net(X,y,theta,alpha,r):
    m = len(y)
    predictions = X @ theta
    mse = (1/(2 * m)) * np.sum((predictions - y)**2)
    
    l1_penalty = r * alpha * np.sum(np.abs(theta))
    l2_penalty = ((1 - r)/2) * alpha * np.sum(theta**2)
    
    return mse + l1_penalty + l2_penalty

def elastic_net_coordinate_descent(X, y, alpha=0.1, r=0.5, iterations=1000):
    m, n = X.shape
    theta = np.zeros(n)

    for it in range(iterations):
        for j in range(n):
            X_j = X[:, j]
            residual = y - (X @ theta) + (theta[j] * X_j)
            rho = X_j @ residual

            # Apply lasso and ridge
            if rho < -r * alpha:
                theta[j] = (rho + r * alpha) / (X_j @ X_j + (1 - r)*alpha)
            elif rho > r * alpha:
                theta[j] = (rho - r * alpha) / (X_j @ X_j + (1 - r)*alpha)
            else:
                theta[j] = 0

        if it % 1000 == 0:  # Print cost every 100 iterations
            cost = compute_cost_elastic_net(X, y, theta, alpha, r)
            print(f"Iteration {it}, Cost {cost}")

    return theta

X_train_bias = add_bias(X_train)
X_test_bias = add_bias(X_test)

alpha = 0.00001  #regularization strength
r = 0.5  # Balance between lasso and ridge regression
iterations = 10000  # Number of iterations

theta_elastic_net = elastic_net_coordinate_descent(X_train_bias, y_train, alpha, r, iterations)
print("\nFinal Theta Values:", theta_elastic_net)


Iteration 0, Cost 28.813833804545563
Iteration 1000, Cost 10.808909375447724
Iteration 2000, Cost 10.808832940842123
Iteration 3000, Cost 10.808831106980895
Iteration 4000, Cost 10.808831020370397
Iteration 5000, Cost 10.808831016060186
Iteration 6000, Cost 10.808831015845106
Iteration 7000, Cost 10.808831015834373
Iteration 8000, Cost 10.808831015833839
Iteration 9000, Cost 10.808831015833814

Final Theta Values: [ 3.01449486e+01 -1.13053280e-01  3.07558019e-02  3.83746624e-02
  2.78643096e+00 -1.70053742e+01  4.43606083e+00 -5.98886112e-03
 -1.44795640e+00  2.64768450e-01 -1.08061031e-02 -9.13260976e-01
  1.23438183e-02 -5.08514343e-01]


In [258]:
#calcualting prediction
y_train_pred = X_train_bias @ theta_elastic_net
y_test_pred = X_test_bias @ theta_elastic_net

#mse calculation
mse_train = np.mean((y_train - y_train_pred)**2)
mse_test = np.mean((y_test - y_test_pred)**2)
print(f'mse_train: {mse_train}')
print(f'mse_test: {mse_test}')

mse_train: 21.610941753657006
mse_test: 24.396858807810347


# Choosing Model

 Even though my Linear Regression  model returns somewhat similar mse value for the test data, I would choose Elastic Net Regression for this dataset to predict future housing prices. It is because it combines the best of Lasso and Ridge Regression model. It helps select important features like Lasso while handling correlated data better than Lasso alone. On the other hand, unlike Ridge, it also allows some coefficients to be exactly zero, making the model more interpretable. Therefore, with learning rate = 0.00001, and 10,000 iterations, this model balances simplicity and accuracy, reducing overfitting while keeping key predictors of this dataset.